## 基础知识

#### Multi Schema

##### 四种状态

- 全局状态：图内部主要使用的状态，创建 StateGraph 时传递给 state_schema 参数。
- 输入状态：图对外接受输入时使用的状态，input_schema 参数。
- 输出状态：图最终对外返回结果时使用的状态，output_schema 参数。
- 私有状态：图内部节点之间传递的临时状态。

##### 设计规范

1. 输入状态和输出状态通常应是全局状态的子集
2. 私有状态和全局状态应尽量避免字段重名
3. 节点函数应明确声明入参状态类型和返回状态类型
4. 节点函数返回的字典应尽量和返回状态类型保持一致

In [ ]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class InputState(TypedDict):
    username: str

class OutputState(TypedDict):
    graph_output: str

class OverallState(TypedDict):
    username: str
    nickname: str
    graph_output: str

class PrivateState(TypedDict):
    greeting: str

def node_1(state: InputState) -> OverallState:
    return {
        "nickname": "Dear " + state["username"],
    }

def node_2(state: OverallState) -> PrivateState:
    print(state)
    return {
        "greeting": "Hello, " + state["nickname"] + "!",
    }

def node_3(state: PrivateState) -> OutputState:
    return {
        "graph_output": state["greeting"],
    }


builder = StateGraph(state_schema=OverallState, input_schema=InputState, output_schema=OutputState)
builder.add_node(node_1)
builder.add_node(node_2)
builder.add_node(node_3)
builder.add_edge(START, 'node_1')
builder.add_edge('node_1', 'node_2')
builder.add_edge('node_2', 'node_3')
builder.add_edge('node_3', END)
graph = builder.compile()
result = graph.invoke({"username": "Max"})
print(result)
